# Limpieza de Datos – Marine

## 1. Importación de paqueterías 


In [94]:
import os
import pandas as pd
from tabulate import tabulate
from unidecode import unidecode
import re
import numpy as np
from rapidfuzz import fuzz, process
from collections import defaultdict, Counter


## 2. Carga de datos

In [95]:
#Carga de archivo
# Archivo original
archivo_original = "C:/Users/KOT14/Documents/Integral/Marine/Procesados/202512/Final/202512_Siniestros_Marine.xlsx"

# Cargar archivo Excel o CSV
df = pd.read_excel(archivo_original)   # o pd.read_csv("datos.csv")
#archivoreadme= "C:\Users\KOT14\OneDrive - Kot Insurance Company AG\Vida\Vida 2025\readme.txt"

### Extracción del nombre del archivo


In [96]:
# Extraer año y mes del nombre del archivo
nombre_base = os.path.basename(archivo_original)  # '202508_Siniestros_Vida.xlsx'
ano_mes = nombre_base[:6]                        # '202508'
ano = ano_mes[:4]                                # '2025'
mes = ano_mes[4:]                                # '08'
mes_num = int(ano_mes[4:])                        # 8 (convertido a entero)

# Diccionario de meses en español
meses = {
    1: "enero", 2: "febrero", 3: "marzo", 4: "abril", 5: "mayo", 6: "junio",
    7: "julio", 8: "agosto", 9: "septiembre", 10: "octubre", 11: "noviembre", 12: "diciembre"
}

nombre_archivo_salida = f"{ano_mes}_Siniestros_Marine_PROCESADO.xlsx"

# Carpeta y nombre de archivo de salida
#carpeta_salida = f"C:/Users/KOT14/OneDrive - Kot Insurance Company AG/06-RM - 02-Actuarial/6. DATABASE/03 Transporte, Carga & Embarcaciones/02 Bases Actuarial/{ano_mes}"
carpeta_salida2 = f"C:/Users/KOT14/OneDrive - Kot Insurance Company AG/Transporte, Carga y Embarcaciones/{ano_mes}"

# Crear carpeta si no existe
#os.makedirs(carpeta_salida, exist_ok=True)
os.makedirs(carpeta_salida2, exist_ok=True)

# Quitar filas completamente vacías
df = df.dropna(how="all")

#Agregar columna con mes
df["MES CARGA"] = f"{mes}/{ano}" 



## 3. Limpieza de datos

### 3.1 Reemplazo y relleno de datos


In [97]:
# ================================
# LIMPIEZA DE DATOS
# ================================
df.loc[df["INWARD POLICY N°"] == "3.6121E+12", "INWARD POLICY N°"] = "'3612100000008"
df.loc[df["INWARD POLICY N°"] == "3612100000008", "INWARD POLICY N°"] = "'3612100000008"
df.loc[df["DESCRIPTION"] == "-", "DESCRIPTION"] = "NO ESPECIFICADO"
df["INWARD POLICY N°"] = df["INWARD POLICY N°"].astype(str)

# Reemplazar valores específicos en COVER
df.loc[df["COVER"] == "CARGA", "COVER"] = "CARGA POLIETILENO"
df.loc[df["COVER"] == "Carga", "COVER"] = "CARGA POLIETILENO"
df.loc[df["COVER"] == "CARGO", "COVER"] = "CARGA POLIETILENO"
df.loc[df["COVER"] == "FletadoresPMI", "COVER"] = "RC FLETADORES(PMI)"
df.loc[df["COVER"] == "Casco y Maquinaria", "COVER"] = "CASCO Y MAQ."
df.loc[df["COVER"] == "CASCO", "COVER"] = "CASCO Y MAQ."
df.loc[df["COVER"] == "Daños a la Maquinaria", "COVER"] = "CASCO Y MAQ."
df.loc[df["COVER"] == "Fletadores RC PMI", "COVER"] = "RC FLETADORES(PMI)"
df.loc[df["COVER"] == "Pandi", "COVER"] = "PANDI"
df.loc[df["COVER"] == "RC Fletadores", "COVER"] = "RC FLETADORES"
df.loc[df["COVER"] == "Transporte", "COVER"] = "TRANSPORTE"
df.loc[df["COVER"] == "Aguas Profundas", "COVER"] = "AGUAS PROFUNDAS"

# Reemplazar valores específicos en Lob-Inward
df["LoB-Inward"] = df["LoB-Inward"].astype(str)
df.loc[df["LoB-Inward"] == "CARGA POLIETILENO", "LoB-Inward"] = "CARGO"
mask = df["LoB-Inward"].str.strip().str.upper() == "FLETADORES PMI"
df.loc[mask, "LoB-Inward"] = "RC FLETADORES(PMI)"

# Rellenar valores nulos en una columna con un valor fijo
df["COVER"] = df["COVER"].fillna("NO ESPECIFICADO")
df["COVERAGE"] = df["COVERAGE"].fillna("NO ESPECIFICADO")
df["STATUS"] = df["STATUS"].fillna("NO ESPECIFICADO")
df["REF. PEMEX"] = df["REF. PEMEX"].fillna("NO ESPECIFICADO")
df["LOCATION"] = df["LOCATION"].fillna("NO ESPECIFICADO")
df["DESCRIPTION"] = df["DESCRIPTION"].fillna("NO ESPECIFICADO")
df["LoB-Inward"] = df["LoB-Inward"].fillna(df['COVER'])
df.loc[df["LoB-Inward"] == "nan", 'LoB-Inward'] = df['COVER']
df["CLAIMS-ID"] = df["CLAIMS-ID"].fillna(df["KEY LOB"])

#Eliminar columnas que no necesitamos
df = df.drop(['NOTAS', 'REF. PEMEX'], axis=1)

# ================================
# DESCRIPTION
# ================================
df["DESCRIPTION"] = (
    df["DESCRIPTION"]
    .astype(str)
    .str.upper()
    .str.strip()
    .apply(lambda x: re.sub(r'[\'\"\[\]\(\)\{\}\{•}\{*}\{-}\{"}]', '', x))  # quita símbolos pero respeta Ñ/Á/É...
)

# ================================
# STATUS
# ================================
#df['STATUS ANTERIOR']=df['STATUS']
df.loc[(df["OSLR Inward"] == 0) & (df["Cumulative CLAIMS PAID"] > 0), "STATUS"] = "P"
df.loc[(df["OSLR Inward"] == 0) & (df["Cumulative CLAIMS PAID"] == 0), "STATUS"] = "C"
#df.loc[(df["OSLR Inward"] > 0) & (df["Cumulative CLAIMS PAID"] > 0), "STATUS"] = "T"  #Suposición mía

In [98]:
print(f"Registros después de limpieza básica: {len(df)}")

Registros después de limpieza básica: 4279


### 3.2 Normalización para Location


In [99]:
# ================================
# LOCATION
# ================================
#region LOCATION
municipios_path = r"C:/Users/KOT14/OneDrive - Kot Insurance Company AG/Códigos/Códigos Transporte/municipios_mexico.xlsx"
df_mun = pd.read_excel(municipios_path)

df_mun["mun_clean"] = df_mun["MUNICIPIO"].astype(str).apply(lambda x: unidecode(x).upper().strip())
df_mun["est_clean"] = df_mun["ESTADO"].astype(str).apply(lambda x: unidecode(x).upper().strip())
municipios_dict = dict(zip(df_mun["mun_clean"], df_mun["est_clean"]))

ciudades_path = "C:/Users/KOT14/OneDrive - Kot Insurance Company AG/Códigos/Códigos Transporte/ciudades_mexico.xlsx"
df_ciu = pd.read_excel(ciudades_path)

df_ciu["city_clean"] = df_ciu["CIUDAD"].astype(str).apply(lambda x: unidecode(x).upper().strip())
df_ciu["estado_clean"] = df_ciu["ESTADO"].astype(str).apply(lambda x: unidecode(x).upper().strip())

ciudades_dict = dict(zip(df_ciu["city_clean"], df_ciu["estado_clean"]))

#autopistas
autopistas = {
    "MEX-QRO": "QUERÉTARO",
    "ARCO NORTE": "HIDALGO",
    "LA RUMOROSA": "BAJA CALIFORNIA",
    "CORDOBA": "VERACRUZ",
    "CORDOBA-VERACRUZ": "VERACRUZ",
    "TUXPAN": "VERACRUZ",
}

#estados de mx con acento
estados_con_acento = {
    "AGUASCALIENTES": "AGUASCALIENTES",
    "BAJA CALIFORNIA": "BAJA CALIFORNIA",
    "BAJA CALIFORNIA SUR": "BAJA CALIFORNIA SUR",
    "CAMPECHE": "CAMPECHE",
    "CHIAPAS": "CHIAPAS",
    "CHIHUAHUA": "CHIHUAHUA",
    "CIUDAD DE MEXICO": "CIUDAD DE MÉXICO",
    "COAHUILA": "COAHUILA",
    "COLIMA": "COLIMA",
    "DURANGO": "DURANGO",
    "ESTADO DE MEXICO": "ESTADO DE MÉXICO",
    "GUANAJUATO": "GUANAJUATO",
    "GUERRERO": "GUERRERO",
    "HIDALGO": "HIDALGO",
    "JALISCO": "JALISCO",
    "MICHOACAN": "MICHOACÁN",
    "MORELOS": "MORELOS",
    "NAYARIT": "NAYARIT",
    "NUEVO LEON": "NUEVO LEÓN",
    "OAXACA": "OAXACA",
    "PUEBLA": "PUEBLA",
    "QUERETARO": "QUERÉTARO",
    "QUINTANA ROO": "QUINTANA ROO",
    "SAN LUIS POTOSI": "SAN LUIS POTOSÍ",
    "SINALOA": "SINALOA",
    "SONORA": "SONORA",
    "TABASCO": "TABASCO",
    "TAMAULIPAS": "TAMAULIPAS",
    "TLAXCALA": "TLAXCALA",
    "VERACRUZ": "VERACRUZ",
    "YUCATAN": "YUCATÁN",
    "ZACATECAS": "ZACATECAS"
}

# abreviaturas de estados
estados_abrev = {
    "AGS": "AGUASCALIENTES",
    "BC": "BAJA CALIFORNIA",
    "BCS": "BAJA CALIFORNIA SUR",
    "CAMP": "CAMPECHE",
    "CHIS": "CHIAPAS",
    "CHIH": "CHIHUAHUA",
    "COAH": "COAHUILA",
    "COL": "COLIMA",
    "CDMX": "CIUDAD DE MÉXICO",
    "D.F.": "CIUDAD DE MÉXICO",
    "D.F": "CIUDAD DE MÉXICO",
    "DGO": "DURANGO",
    "GTO": "GUANAJUATO",
    "GRO": "GUERRERO",
    "HGO": "HIDALGO",
    "JAL": "JALISCO",
    "EDO MEX": "ESTADO DE MÉXICO",
    "EDO DE MEX": "ESTADO DE MÉXICO",
    "EDO DE MEXICO": "ESTADO DE MÉXICO",
    "EDOMEX": "ESTADO DE MÉXICO",
    "MEX": "ESTADO DE MÉXICO",
    "MICH": "MICHOACÁN",
    "MOR": "MORELOS",
    "NAY": "NAYARIT",
    "NL": "NUEVO LEÓN",
    "NL.": "NUEVO LEÓN",
    "N.L": "NUEVO LEÓN",
    "N.L.": "NUEVO LEÓN",
    "OAX": "OAXACA",
    "PUE": "PUEBLA",
    "QRO": "QUERÉTARO",
    "QROO": "QUINTANA ROO",
    "SLP": "SAN LUIS POTOSÍ",
    "SIN": "SINALOA",
    "SON": "SONORA",
    "TAB": "TABASCO",
    "TAM": "TAMAULIPAS",
    "TLAX": "TLAXCALA",
    "VER": "VERACRUZ",
    "VERACRZ": "VERACRUZ",
    "TAD PAJARITOS": "VERACRUZ",
    "YUC": "YUCATÁN",
    "ZAC": "ZACATECAS",
}

# ================================
# FUNCIONES PREVIAS A BUSCAR ESTADO
# ================================
def clean_keep_accents(texto):
    if pd.isna(texto):
        return ""
    texto = str(texto).upper()
    texto = re.sub(r"[^A-ZÁÉÍÓÚÜÑ0-9\s.,-]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

def fuzzy_match(lista, texto, threshold=85):
    match = process.extractOne(texto, lista, scorer=fuzz.partial_ratio)
    if match and match[1] >= threshold:
        return match[0]
    return None

def load_municipios(municipios_path):
    df_mun = pd.read_excel(municipios_path)
    df_mun["MUNICIPIO_CLEAN"] = df_mun["MUNICIPIO"].str.upper()
    df_mun["ESTADO_CLEAN"] = df_mun["ESTADO"].str.upper()
    municipios_dict = dict(zip(df_mun["MUNICIPIO_CLEAN"], df_mun["ESTADO_CLEAN"]))
    estados_set = set(df_mun["ESTADO_CLEAN"])
    return municipios_dict, estados_set

# ================================
# FUNCIÓN PRINCIPAL DE MÉXICO Y USA
# ================================
usa_keywords = [
    "USA", "UNITED STATES", "EUA", "EEUU",
    "TEXAS", "HOUSTON", "GALVESTON", "BEAUMONT", "PORT ARTHUR"
]
usa_pattern = re.compile("|".join(usa_keywords), re.IGNORECASE)

estados_mx = list(estados_abrev.values())
estados_mx_pattern = re.compile("|".join(estados_mx), re.IGNORECASE)

def es_usa(texto):
    t = clean_keep_accents(texto)
    if usa_pattern.search(t):
        if estados_mx_pattern.search(t):
            return False
        return True
    return False

municipios_map = defaultdict(set)
for mun, est in zip(df_mun["mun_clean"], df_mun["est_clean"]):
    municipios_map[mun].add(est)

mun_list = sorted(municipios_map.keys(), key=len, reverse=True)
mun_pattern = re.compile(r"\b(" + "|".join(re.escape(m) for m in mun_list) + r")\b")

ciu_list = sorted(ciudades_dict.keys(), key=len, reverse=True)
ciu_pattern = re.compile(r"\b(" + "|".join(re.escape(c) for c in ciu_list) + r")\b")

auto_pattern = re.compile(r"\b(" + "|".join(re.escape(a) for a in autopistas.keys()) + r")\b")

abrev_pattern = re.compile(r"\b(" + "|".join(re.escape(a) for a in estados_abrev.keys()) + r")\b")

states_set = set(estados_abrev.values())

# ================================
# FUNCIÓN ROBUSTA - retorna tupla (mun, estado, auto)
# ================================
def get_municipio_estado(texto):
    t = clean_keep_accents(texto)

    if es_usa(t):
        return (None, "USA", None)

    m = mun_pattern.search(t)
    if m:
        mun = m.group(1)
        estados = list(municipios_map[mun])
        if len(estados) == 1:
            return (mun, estados[0], None)
        for e in estados:
            if e in t:
                return (mun, e, None)
        return (mun, estados[0], None)

    m = auto_pattern.search(t)
    if m:
        auto = m.group(1)
        return (auto, autopistas[auto], auto)

    m = ciu_pattern.search(t)
    if m:
        city = m.group(1)
        return (city, ciudades_dict[city], None)

    m = abrev_pattern.search(t)
    if m:
        ab = m.group(1)
        return (None, estados_abrev[ab], None)
    
    candidatos = []
    for est in estados_con_acento:
        idx = t.find(est)
        if idx != -1:
            candidatos.append((idx, est))

    tokens = t.split()
    for tok in tokens:
        if tok in estados_abrev:
            idx = t.find(tok)
            candidatos.append((idx, estados_abrev[tok]))

    if candidatos:
        candidatos.sort(key=lambda x: x[0])
        return (None, candidatos[0][1], None)

    return (None, "NO ESPECIFICADO", None)

# ================================
# APLICAR AL DATAFRAME - versión robusta
# ================================
print("Antes de apply - columnas:", len(df.columns))

# Aplicar y desempacar tupla en 3 columnas separadas
# NOTA: se usa zip para evitar pd.Series por fila, que es más lento
# y no afecta el índice original del df
municipios_res, estados_res, autopistas_res = zip(
    *df['LOCATION'].apply(get_municipio_estado)
)
df['MUNICIPIO'] = municipios_res
df['ESTADO']    = estados_res
df['AUTOPISTA'] = autopistas_res

print("Después de apply - columnas:", len(df.columns))

df["LOCATION"] = df["LOCATION"].apply(clean_keep_accents)
df = df.drop(['MUNICIPIO', 'AUTOPISTA'], axis=1)
df["ESTADO"] = df["ESTADO"].str.upper().map(estados_con_acento).fillna(df["ESTADO"])
df.loc[df["LOCATION"] == "NO ESPECIFICADO", "ESTADO"] = "NO ESPECIFICADO"
df["ESTADO"] = df["ESTADO"].fillna("NO ESPECIFICADO")
#endregion


Antes de apply - columnas: 37
Después de apply - columnas: 40


In [100]:
print(f"Registros después de normalización Location: {len(df)}")

Registros después de normalización Location: 4279


### 3.3 Tratamiento de valores duplicados


In [101]:
# ================================
# MANEJO DE LOS DUPLICADOS
# ================================
#region DUPLICADOS
id_col = "CLAIM NUMBER"

def rellenar_informacion_grupo(grupo):
    """
    Completa LOCATION y DESCRIPTION faltantes usando datos del mismo siniestro.
    """
    locations_validas = grupo[grupo["LOCATION"] != "NO ESPECIFICADO"]["LOCATION"].unique()
    if len(locations_validas) > 0:
        grupo.loc[grupo["LOCATION"] == "NO ESPECIFICADO", "LOCATION"] = locations_validas[0]

    descriptions_validas = grupo[grupo["DESCRIPTION"] != "NO ESPECIFICADO"]["DESCRIPTION"].unique()
    if len(descriptions_validas) > 0:
        grupo.loc[grupo["DESCRIPTION"] == "NO ESPECIFICADO", "DESCRIPTION"] = descriptions_validas[0]

    estados_validos = grupo[grupo["ESTADO"] != "NO ESPECIFICADO"]["ESTADO"].unique()
    if len(estados_validos) > 0:
        grupo.loc[grupo["ESTADO"] == "NO ESPECIFICADO", "ESTADO"] = estados_validos[0]

    return grupo

def unir_no_especificado(grupo):
    if len(grupo) == 2:
        fila_buena = grupo[
            (grupo["LOCATION"] != "NO ESPECIFICADO") &
            (grupo["GROSS RESERVE"] > 0) &
            (grupo["DEDUCTIBLE"] > 0) &
            (grupo["DESCRIPTION"] != "NO ESPECIFICADO")
        ]
        if len(fila_buena) == 1:
            return fila_buena
    return grupo

num_cols = [
    'Cumulative EXPENSES PAID',
    'Cumulative VALUATION EXPENSES',
    'Cumulative CLAIMS PAID',
    'Total Claims Paid Inward',
    'Total Claims Paid no Alae',
    'Inward Incurred',
    'Inward Incurred no Alae',
    'GROSS RESERVE',
    'DEDUCTIBLE'
]

def seleccionar_mejores_registros(grupo):
    grupo = grupo.copy()

    # 0️⃣ PAGOS REALES GANAN TODO
    if "Cumulative CLAIMS PAID" in grupo.columns and len(grupo) > 1:
        con_paid = grupo[grupo["Cumulative CLAIMS PAID"].fillna(0) > 0]
        sin_paid = grupo[grupo["Cumulative CLAIMS PAID"].fillna(0) == 0]

        if not con_paid.empty and not sin_paid.empty:
            ded_con_paid = set(con_paid["DEDUCTIBLE"].fillna(0).unique())
            sin_paid_con_ded_diff = sin_paid[
                (~sin_paid["DEDUCTIBLE"].fillna(0).isin(ded_con_paid)) &
                (sin_paid["DEDUCTIBLE"].fillna(0) > 0)
            ]
            return pd.concat([con_paid, sin_paid_con_ded_diff])
        elif not con_paid.empty:
            return con_paid

    # 1️⃣ DEDUCIBLE
    if "DEDUCTIBLE" in grupo.columns and len(grupo) > 1:
        cols_compare = [c for c in grupo.columns if c != "DEDUCTIBLE"]
        dup = grupo.duplicated(subset=cols_compare, keep=False)
        if dup.any():
            candidatos = grupo[dup]
            con_ded = candidatos[candidatos["DEDUCTIBLE"].fillna(0) > 0]
            if not con_ded.empty:
                grupo = con_ded[con_ded["DEDUCTIBLE"] == con_ded["DEDUCTIBLE"].max()]

    # 2️⃣ DEFINIR FINANZAS
    _num_cols = grupo.select_dtypes(include="number").columns
    tiene_finanzas_fila = grupo[_num_cols].fillna(0).sum(axis=1) > 0
    ambos_tienen_finanzas = tiene_finanzas_fila.all()

    # 3️⃣ PAGOS
    cols_pagos = [c for c in ["Cumulative EXPENSES PAID", "Cumulative VALUATION EXPENSES",
                               "Cumulative CLAIMS PAID"] if c in grupo.columns]
    tiene_reserve_o_ded = (
        (grupo["DEDUCTIBLE"].fillna(0) > 0) | (grupo["GROSS RESERVE"].fillna(0) > 0)
    ).any()

    if cols_pagos and len(grupo) > 1 and not tiene_reserve_o_ded:
        tiene_pagos = grupo[cols_pagos].fillna(0).gt(0).any(axis=1)
        if tiene_pagos.any():
            grupo = grupo[tiene_pagos]

    # 4️⃣ ELIMINAR STATUS = C SIN RESERVE NI DED
    if len(grupo) > 1 and ambos_tienen_finanzas:
        eliminar = (
            (grupo["GROSS RESERVE"].fillna(0) == 0) &
            (grupo["DEDUCTIBLE"].fillna(0) == 0) &
            (grupo["STATUS"] == "C")
        )
        if eliminar.any() and (~eliminar).any():
            grupo = grupo[~eliminar]

    # 5️⃣ STATUS P
    if "STATUS" in grupo.columns and len(grupo) > 1 and not ambos_tienen_finanzas:
        status_p = grupo[grupo["STATUS"] == "P"]
        if not status_p.empty:
            grupo = status_p

    # 6️⃣ FILAS CON NÚMEROS
    if len(_num_cols) > 0 and len(grupo) > 1:
        con_numeros = grupo[_num_cols].notna().any(axis=1)
        if con_numeros.any():
            grupo = grupo[con_numeros]

    # 7️⃣ COMPLETITUD
    completitud = grupo.notna().mean(axis=1)
    grupo = grupo.loc[completitud == completitud.max()]

    # 8️⃣ DESEMPATE FINAL
    if len(grupo) > 1 and len(_num_cols) > 0:
        nums = grupo[_num_cols].fillna(0)
        if nums.nunique().max() == 1:
            grupo = grupo.sort_index().tail(1)

    return grupo

# ================================
# APLICAR LOS TRES GROUPBY
# La clave: set_index antes → groupby opera sobre las demás columnas
# → reset_index al final devuelve CLAIM NUMBER como columna normal
# ================================
df = (
    df
    .set_index(id_col)
    .groupby(level=0, group_keys=False)
    .apply(rellenar_informacion_grupo)
    .reset_index()
)

df = (
    df
    .set_index(id_col)
    .groupby(level=0, group_keys=False)
    .apply(unir_no_especificado)
    .reset_index()
)

df = (
    df
    .set_index(id_col)
    .groupby(level=0, group_keys=False)
    .apply(seleccionar_mejores_registros)
    .reset_index()
)

print(f"✓ {id_col} presente: {id_col in df.columns}")

# ================================
# ELIMINAR FILAS CON TODOS CEROS
# ================================
claims_con_cero = (
    df.groupby(id_col)
      .apply(lambda g: (g[num_cols].fillna(0).sum(axis=1) == 0).sum() == 1 and len(g) == 2)
)

df["ELIMINADO_POR_CEROS"] = df[id_col].isin(claims_con_cero[claims_con_cero].index)
df = df.drop_duplicates()
df = df.drop(columns="ELIMINADO_POR_CEROS")

# ================================
# REORDENAR COLUMNAS: CLAIM NUMBER después de CLAIMS-ID
# ================================
cols = df.columns.tolist()
if 'CLAIM NUMBER' in cols and 'CLAIMS-ID' in cols:
    cols.remove('CLAIM NUMBER')
    idx = cols.index('CLAIMS-ID')
    cols.insert(idx + 1, 'CLAIM NUMBER')
    df = df[cols]
    print("✓ CLAIM NUMBER movido después de CLAIMS-ID")
#endregion


✓ CLAIM NUMBER presente: True
✓ CLAIM NUMBER movido después de CLAIMS-ID


## 4. Exportación de resultados


In [102]:
# Guardar Excel
#ruta_salida = os.path.join(carpeta_salida, nombre_archivo_salida)
ruta_salida2 = os.path.join(carpeta_salida2, nombre_archivo_salida)

with pd.ExcelWriter(ruta_salida2, engine="xlsxwriter") as writer:
    df.to_excel(writer, sheet_name="Hoja1", index=False)
    
    workbook  = writer.book
    worksheet = writer.sheets["Hoja1"]
    formato_pct = workbook.add_format({'num_format': '0%'})  # formato porcentaje sin decimales
    col_idx = df.columns.get_loc("KOT SHARE")
    worksheet.set_column(col_idx, col_idx, None, formato_pct)

#print(f"Archivo guardado en: {ruta_salida}")  
print(f"Archivo guardado en: {ruta_salida2}")

Archivo guardado en: C:/Users/KOT14/OneDrive - Kot Insurance Company AG/Transporte, Carga y Embarcaciones/202512\202512_Siniestros_Marine_PROCESADO.xlsx
